# 08 통합 Manifest — 멀티 파이프라인 결과 병합

**03~07 노트북**에서 생성한 개별 manifest JSON을 **프레임 단위로 병합**해 하나의 `unified_manifest.json`을 만듭니다.

| Part | 주제 |
|------|------|
| 1 | manifest 소스 탐색 · IPAD 프레임 인덱스 |
| 2 | 소스별 로드 · 정규화(normalize) |
| 3 | 프레임 키 조인 · 신호(signal) 병합 |
| 4 | 통합 manifest export · 대시보드 시각화 |

### 통합 대상 manifest

| 노트북 | 파일 | 주요 신호 |
|--------|------|-----------|
| 03 샘플링 | `output_sampling/sampling_manifest.json` | swin/detr 선별 프레임 인덱스 |
| 05 VLM 요약 | `output_vlm_summary/summary_manifest.json` | vlm_frames, text_summary |
| 04 이상 탐지 | `output_anomaly_scene/anomaly_manifest.json` | anomaly_scenes, top_frame_indices |
| 06 OCR | `output_ocr_kr/ocr_manifest_ipad.json` | full_text, detections |
| 07 키포인트 | `output_keypoint/keypoint_manifest.json` | displacement, motion_z |

```
개별 manifest (03~07)
        ↓ normalize + frame key
   프레임별 signal 레코드
        ↓ join
unified_manifest.json + figures/
```

> 선행 실행: 통합하려는 노트북(03~07)을 먼저 실행해 각 `output_*` 폴더에 manifest를 생성하세요. 없어도 **있는 manifest만** 병합합니다.


### 본 실습에서 다루는 기술

| 기법 | 한 줄 설명 | 코드에서의 역할 |
|------|-----------|----------------|
| **Manifest Registry** | 소스 경로·스키마 정의 | `MANIFEST_SOURCES` |
| **Frame Key** | 경로/인덱스 통일 키 | `normalize_frame_key`, `frame_key_from_index` |
| **Signal Extractor** | 소스별 필드 파싱 | `extract_*_signals` |
| **Join Engine** | 프레임 단위 병합 | `merge_frame_records` |
| **Unified Export** | 최종 JSON | `export_unified_manifest` |
| **Coverage Dashboard** | 소스별 커버리지 | `plot_manifest_coverage` |

핵심: 영상 AI 파이프라인은 **모듈별 JSON**을 만들고, 실무 연동은 **통합 manifest** 한 장으로 합니다.


In [ ]:
# ── 패키지 설치 (최초 1회, 주석 해제 후 실행) ──
# !pip install -q -r requirements.txt


In [ ]:
from __future__ import annotations

# =============================================================================
# 환경 설정 — import, manifest 소스 경로, 출력 경로, 한글 폰트
# =============================================================================

import json
import os
import platform
import subprocess
import zipfile
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# ── 개별 manifest 입력 경로 (03~07 산출물) ───────────────────────────────────
MANIFEST_SOURCES = {
    "sampling": Path("output_sampling/sampling_manifest.json"),
    "vlm_summary": Path("output_vlm_summary/summary_manifest.json"),
    "anomaly_scene": Path("output_anomaly_scene/anomaly_manifest.json"),
    "ocr_ipad": Path("output_ocr_kr/ocr_manifest_ipad.json"),
    "ocr_hub": Path("output_ocr_kr/ocr_manifest.json"),
    "keypoint": Path("output_keypoint/keypoint_manifest.json"),
}

# ── 통합 manifest 출력 ───────────────────────────────────────────────────────
OUTPUT_DIR = Path("output_unified")
FIG_DIR = OUTPUT_DIR / "figures"
UNIFIED_MANIFEST_PATH = OUTPUT_DIR / "unified_manifest.json"
COVERAGE_REPORT_PATH = OUTPUT_DIR / "manifest_coverage.json"

# IPAD 프레임 (조인 기준 타임라인)
IPAD_ZIP_NAMES = ("IPAD_sample.zip", "IPAD_Sample.zip")
IPAD_SEARCH_PATHS = [Path("IPAD_Sample"), Path("IPAD_sample")]
FRAME_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
FPS_ASSUMED = 30.0

for d in (OUTPUT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)


def is_running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def setup_matplotlib_korean() -> None:
    from matplotlib import font_manager
    if is_running_in_colab():
        subprocess.run(
            ["apt-get", "-qq", "-y", "install", "fonts-nanum"],
            check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        font_manager._load_fontmanager(try_read_cache=False)
    if platform.system() == "Windows":
        candidates = ["Malgun Gothic", "NanumGothic", "Gulim"]
        fallback = Path(os.environ.get("WINDIR", r"C:\Windows")) / "Fonts" / "malgun.ttf"
    elif platform.system() == "Darwin":
        candidates = ["AppleGothic", "Apple SD Gothic Neo"]
        fallback = None
    else:
        candidates = ["NanumGothic", "NanumBarunGothic"]
        fallback = Path("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    available = {f.name for f in font_manager.fontManager.ttflist}
    chosen = next((n for n in candidates if n in available), None)
    if chosen:
        plt.rcParams["font.family"] = chosen
    elif fallback and fallback.exists():
        font_manager.fontManager.addfont(str(fallback))
        plt.rcParams["font.family"] = font_manager.FontProperties(fname=str(fallback)).get_name()
    plt.rcParams["axes.unicode_minus"] = False


@dataclass
class LoadedManifest:
    name: str
    path: Path
    data: dict
    status: str = "ok"


@dataclass
class FrameRecord:
    frame_index: int | None = None
    image_path: str | None = None
    frame_key: str = ""
    signals: dict = field(default_factory=dict)
    sources: list[str] = field(default_factory=list)


setup_matplotlib_korean()
print("환경 설정 완료")


In [ ]:
# =============================================================================
# IPAD 프레임 로드 · manifest I/O · frame key 유틸
# =============================================================================

def _search_roots() -> list[Path]:
    roots: list[Path] = [Path.cwd().resolve()]
    try:
        from IPython import get_ipython
        ip = get_ipython()
        nb_file = ip.user_ns.get("__vsc_ipynb_file__") if ip else None
        if nb_file:
            nb_dir = Path(nb_file).resolve().parent
            if nb_dir not in roots:
                roots.insert(0, nb_dir)
    except Exception:
        pass
    return roots


def find_ipad_zip() -> Path | None:
    for root in _search_roots():
        for name in IPAD_ZIP_NAMES:
            p = root / name
            if p.is_file():
                return p
    return None


def ensure_ipad_sample() -> Path | None:
    for root in _search_roots():
        for rel in IPAD_SEARCH_PATHS:
            base = root / rel
            if base.is_dir() and any(p.suffix.lower() in FRAME_EXTS for p in base.rglob("*") if p.is_file()):
                return base
    zpath = find_ipad_zip()
    if zpath is None:
        return None
    extract_to = zpath.parent
    print(f"[IPAD] {zpath.name} 압축 해제 -> {extract_to.resolve()}")
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(extract_to)
    for rel in IPAD_SEARCH_PATHS:
        if (extract_to / rel).is_dir():
            return extract_to / rel
    return None


def load_ipad_testing_frames() -> list[Path]:
    # testing 클립 프레임을 정렬해 반환 (통합 타임라인 기준).
    root = ensure_ipad_sample()
    if root is None:
        return []
    for split in ("testing", "training"):
        for frames_root in root.rglob("frames"):
            if split not in str(frames_root):
                continue
            for clip_dir in sorted(frames_root.iterdir()):
                if not clip_dir.is_dir():
                    continue
                files = sorted(
                    [p for p in clip_dir.iterdir() if p.suffix.lower() in FRAME_EXTS],
                    key=lambda p: p.name,
                )
                if files:
                    return files
    return sorted(p for p in root.rglob("*") if p.suffix.lower() in FRAME_EXTS)


def load_json_manifest(path: Path) -> dict | None:
    if not path.is_file():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        print(f"[WARN] JSON 파싱 실패: {path} ({exc})")
        return None


def normalize_frame_key(path_or_name: str) -> str:
    # 프레임 조인 키: 파일명(000.jpg) + clip 경로 tail. frames/.../파일명 형태로 통일.
    p = Path(path_or_name)
    parts = [x.lower() for x in p.parts]
    if "frames" in parts:
        i = parts.index("frames")
        tail = parts[i:]
        return "/".join(tail[-3:]) if len(tail) >= 3 else "/".join(tail)
    return p.name.lower()


def frame_key_from_index(ipad_frames: list[Path], idx: int) -> str | None:
    if 0 <= idx < len(ipad_frames):
        return normalize_frame_key(str(ipad_frames[idx]))
    return None


def index_from_frame_key(ipad_frames: list[Path], key: str) -> int | None:
    lookup = {normalize_frame_key(str(p)): i for i, p in enumerate(ipad_frames)}
    return lookup.get(key)


def sec_from_index(idx: int, fps: float = FPS_ASSUMED) -> float:
    return round(idx / fps, 3)


## Part 1 — manifest 소스 탐색 · IPAD 타임라인

개별 manifest 파일 존재 여부를 확인하고, IPAD testing 프레임 목록을 **공통 타임라인**으로 준비합니다.


In [ ]:
# ── Part 1: manifest 소스 스캔 · IPAD 프레임 ──

loaded: list[LoadedManifest] = []
missing: list[str] = []

for name, path in MANIFEST_SOURCES.items():
    data = load_json_manifest(path)
    if data is None:
        missing.append(name)
        print(f"[MISS] {name:14s} -> {path}")
    else:
        loaded.append(LoadedManifest(name=name, path=path, data=data))
        print(f"[ OK ] {name:14s} -> {path}")

ipad_frames = load_ipad_testing_frames()
print(f"\nIPAD testing 프레임: {len(ipad_frames)}장")
if ipad_frames:
    print(f"  예: {ipad_frames[0].name} ... {ipad_frames[-1].name}")

coverage = {
    "scanned_at": datetime.now(timezone.utc).isoformat(),
    "sources_found": [m.name for m in loaded],
    "sources_missing": missing,
    "ipad_frame_count": len(ipad_frames),
}
COVERAGE_REPORT_PATH.write_text(json.dumps(coverage, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n커버리지 리포트: {COVERAGE_REPORT_PATH}")


## Part 2 — 소스별 신호 추출(normalize)

각 manifest에서 **프레임 단위 신호**만 뽑아 공통 스키마로 변환합니다.


In [ ]:
# ── Part 2: 소스별 signal extractor ──

def _ensure_record(store: dict[str, FrameRecord], key: str) -> FrameRecord:
    if key not in store:
        store[key] = FrameRecord(frame_key=key)
    return store[key]


def extract_sampling_signals(data: dict, store: dict[str, FrameRecord], ipad_frames: list[Path]) -> None:
    presets = data.get("presets", {})
    for preset_name, indices in presets.items():
        for idx in indices:
            key = frame_key_from_index(ipad_frames, int(idx))
            if key is None:
                continue
            rec = _ensure_record(store, key)
            rec.frame_index = int(idx)
            rec.signals[f"sampling_{preset_name}"] = True
            rec.sources.append("sampling")


def extract_vlm_signals(data: dict, store: dict[str, FrameRecord], ipad_frames: list[Path]) -> None:
    for idx in data.get("vlm_frames", []):
        key = frame_key_from_index(ipad_frames, int(idx))
        if key is None:
            continue
        rec = _ensure_record(store, key)
        rec.frame_index = int(idx)
        rec.signals["vlm_keyframe"] = True
        rec.sources.append("vlm_summary")
    # 클립 전체 메타는 clip-level로 store["_clip_meta"]에 보관
    store.setdefault("_clip_meta", FrameRecord(frame_key="_clip_meta"))
    store["_clip_meta"].signals["vlm_text_summary"] = data.get("text_summary")
    store["_clip_meta"].signals["vlm_query"] = data.get("query")
    store["_clip_meta"].signals["vlm_segments"] = data.get("segments")
    store["_clip_meta"].sources.append("vlm_summary")


def extract_anomaly_signals(data: dict, store: dict[str, FrameRecord], ipad_frames: list[Path]) -> None:
    top_idxs = set(data.get("top_frame_indices", []))
    anomaly_idxs = set(data.get("anomaly_frames", []))
    for idx in top_idxs | anomaly_idxs:
        key = frame_key_from_index(ipad_frames, int(idx))
        if key is None:
            continue
        rec = _ensure_record(store, key)
        rec.frame_index = int(idx)
        rec.signals["anomaly_candidate"] = idx in anomaly_idxs
        rec.signals["anomaly_top"] = idx in top_idxs
        rec.sources.append("anomaly_scene")
    store.setdefault("_clip_meta", FrameRecord(frame_key="_clip_meta"))
    store["_clip_meta"].signals["anomaly_scenes"] = data.get("anomaly_scenes", [])
    store["_clip_meta"].signals["anomaly_report"] = data.get("text_report")
    store["_clip_meta"].sources.append("anomaly_scene")


def extract_ocr_signals(data: dict, store: dict[str, FrameRecord], source_name: str) -> None:
    for item in data.get("results", []):
        path = item.get("image_path", "")
        key = normalize_frame_key(path)
        rec = _ensure_record(store, key)
        rec.image_path = path
        rec.signals["ocr_full_text"] = item.get("full_text", "")
        rec.signals["ocr_detection_count"] = len(item.get("detections", []))
        rec.signals["ocr_source"] = source_name
        rec.sources.append(source_name)


def extract_keypoint_signals(data: dict, store: dict[str, FrameRecord]) -> None:
    z_series = data.get("displacement_zscore", [])
    for i, fr in enumerate(data.get("frames", [])):
        path = fr.get("image_path", "")
        key = normalize_frame_key(path)
        rec = _ensure_record(store, key)
        rec.image_path = path
        rec.signals["keypoint_method"] = fr.get("method")
        rec.signals["keypoint_count"] = fr.get("num_feature_points")
        rec.signals["displacement"] = fr.get("displacement")
        rec.signals["motion_z"] = fr.get("motion_z")
        if i > 0 and i - 1 < len(z_series):
            rec.signals.setdefault("motion_z", z_series[i - 1])
        rec.sources.append("keypoint")
    store.setdefault("_clip_meta", FrameRecord(frame_key="_clip_meta"))
    store["_clip_meta"].signals["motion_spike_frames"] = data.get("motion_spike_frames", [])
    store["_clip_meta"].signals["mean_displacement"] = data.get("mean_displacement")
    store["_clip_meta"].sources.append("keypoint")


EXTRACTORS = {
    "sampling": lambda d, s: extract_sampling_signals(d, s, ipad_frames),
    "vlm_summary": lambda d, s: extract_vlm_signals(d, s, ipad_frames),
    "anomaly_scene": lambda d, s: extract_anomaly_signals(d, s, ipad_frames),
    "ocr_ipad": lambda d, s: extract_ocr_signals(d, s, "ocr_ipad"),
    "ocr_hub": lambda d, s: extract_ocr_signals(d, s, "ocr_hub"),
    "keypoint": lambda d, s: extract_keypoint_signals(d, s),
}

frame_store: dict[str, FrameRecord] = {}
for m in loaded:
    fn = EXTRACTORS.get(m.name)
    if fn:
        fn(m.data, frame_store)

# IPAD 타임라인에 존재하지만 신호가 없는 프레임도 빈 레코드로 채움
for i, fp in enumerate(ipad_frames):
    key = normalize_frame_key(str(fp))
    rec = _ensure_record(frame_store, key)
    rec.frame_index = i
    rec.image_path = str(fp)
    rec.signals.setdefault("time_sec", sec_from_index(i))

print(f"추출된 frame_key 수: {len([k for k in frame_store if not k.startswith('_')])}")
print(f"clip-level 메타: {'_clip_meta' in frame_store}")


## Part 3 — 프레임 조인 · 우선순위 · 알람 후보

여러 신호를 합쳐 **프레임별 통합 스코어**와 **알람 후보**를 산출합니다.


In [ ]:
# ── Part 3: 통합 스코어 · 알람 후보 ──

def compute_unified_score(signals: dict) -> float:
    # 간단한 가중 스코어. 실무에서는 도메인별 weight로 교체.
    score = 0.0
    if signals.get("anomaly_candidate"):
        score += 0.45
    if signals.get("vlm_keyframe"):
        score += 0.20
    if signals.get("sampling_swin") or signals.get("sampling_detr"):
        score += 0.10
    mz = signals.get("motion_z")
    if isinstance(mz, (int, float)) and mz >= 2.0:
        score += 0.20
    if signals.get("ocr_detection_count", 0) > 0:
        score += 0.05
    return round(min(score, 1.0), 4)


def build_unified_records(store: dict[str, FrameRecord]) -> list[dict]:
    records = []
    for key, rec in store.items():
        if key.startswith("_"):
            continue
        sig = dict(rec.signals)
        sig["unified_score"] = compute_unified_score(sig)
        records.append({
            "frame_key": rec.frame_key,
            "frame_index": rec.frame_index,
            "image_path": rec.image_path,
            "time_sec": sig.get("time_sec"),
            "sources": sorted(set(rec.sources)),
            "signals": sig,
        })
    records.sort(key=lambda r: (r.get("frame_index") is None, r.get("frame_index", 10**9)))
    return records


def select_alert_candidates(records: list[dict], threshold: float = 0.55, top_k: int = 8) -> list[dict]:
    ranked = sorted(records, key=lambda r: r["signals"]["unified_score"], reverse=True)
    alerts = [r for r in ranked if r["signals"]["unified_score"] >= threshold]
    if len(alerts) < top_k:
        alerts = ranked[:top_k]
    return alerts[:top_k]


unified_records = build_unified_records(frame_store)
alert_candidates = select_alert_candidates(unified_records)

print(f"통합 레코드: {len(unified_records)}")
print(f"알람 후보: {len(alert_candidates)}")
if alert_candidates:
    top = alert_candidates[0]
    print(
        f"  top1 idx={top.get('frame_index')} score={top['signals']['unified_score']:.3f}"
        f" sources={top['sources']}"
    )


## Part 4 — unified_manifest.json export · 커버리지 대시보드

최종 JSON을 저장하고, 프레임별 **소스 커버리지 히트맵**을 시각화합니다.


In [ ]:
# ── Part 4: export · 시각화 ──

def export_unified_manifest(
    records: list[dict],
    alerts: list[dict],
    clip_meta: dict,
    coverage: dict,
) -> Path:
    payload = {
        "version": "1.0",
        "created_at": datetime.now(timezone.utc).isoformat(),
        "description": "03~07 파이프라인 통합 manifest",
        "coverage": coverage,
        "clip_meta": clip_meta,
        "num_frames": len(records),
        "alert_candidates": alerts,
        "frames": records,
    }
    UNIFIED_MANIFEST_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return UNIFIED_MANIFEST_PATH


def plot_manifest_coverage(records: list[dict], save_path: Path, max_rows: int = 40) -> None:
    # 프레임 x 신호 유무 히트맵.
    signal_cols = [
        "sampling_swin", "sampling_detr", "vlm_keyframe",
        "anomaly_candidate", "anomaly_top", "ocr_detection_count",
        "displacement", "motion_z", "unified_score",
    ]
    rows = records[:max_rows]
    if not rows:
        print("[Plot] 레코드 없음")
        return

    mat = np.zeros((len(rows), len(signal_cols)), dtype=np.float32)
    for i, rec in enumerate(rows):
        sig = rec["signals"]
        for j, col in enumerate(signal_cols):
            val = sig.get(col, 0)
            if isinstance(val, bool):
                mat[i, j] = 1.0 if val else 0.0
            elif col == "ocr_detection_count":
                mat[i, j] = 1.0 if (val or 0) > 0 else 0.0
            elif col == "unified_score":
                mat[i, j] = float(val or 0)
            else:
                mat[i, j] = 0.0 if val is None else min(float(val) / 3.0, 1.0)

    fig, ax = plt.subplots(figsize=(12, max(4, len(rows) * 0.22)))
    im = ax.imshow(mat, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
    ax.set_xticks(range(len(signal_cols)))
    ax.set_xticklabels(signal_cols, rotation=35, ha="right")
    ylabels = [str(r.get("frame_index", "?")) for r in rows]
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels(ylabels)
    ax.set_xlabel("신호")
    ax.set_ylabel("frame_index")
    ax.set_title("통합 manifest 커버리지 (프레임 x 신호)")
    fig.colorbar(im, ax=ax, fraction=0.02)
    plt.tight_layout()
    fig.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show()


clip_meta_rec = frame_store.get("_clip_meta")
clip_meta = {
    "signals": clip_meta_rec.signals if clip_meta_rec else {},
    "sources": sorted(set(clip_meta_rec.sources)) if clip_meta_rec else [],
}

manifest_path = export_unified_manifest(
    unified_records,
    alert_candidates,
    clip_meta,
    coverage,
)
plot_manifest_coverage(unified_records, FIG_DIR / "01_coverage_heatmap.png")

# 알람 후보 요약표
if alert_candidates:
    fig, ax = plt.subplots(figsize=(10, 0.35 * len(alert_candidates) + 1.5))
    ax.axis("off")
    lines = ["frame_idx | score | sources | ocr_snippet", "-" * 72]
    for r in alert_candidates:
        ocr = (r["signals"].get("ocr_full_text") or "")[:40].replace("\n", " ")
        lines.append(
            f"{str(r.get('frame_index')):>9} | {r['signals']['unified_score']:.3f} | "
            f"{','.join(r['sources']):<18} | {ocr}"
        )
    ax.text(0.01, 0.99, "\n".join(lines), va="top", family="monospace", fontsize=9)
    ax.set_title("알람 후보 Top frames")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "02_alert_candidates.png", dpi=130, bbox_inches="tight")
    plt.show()

print(f"통합 manifest 저장: {manifest_path}")
print(f"  frames={len(unified_records)} | alerts={len(alert_candidates)}")


---

### 실습 요약

- **Part 1** — 03~07 manifest 존재 확인 · IPAD testing 타임라인
- **Part 2** — 소스별 signal 추출 · frame_key 정규화
- **Part 3** — unified_score · 알람 후보 선정
- **Part 4** — `unified_manifest.json` · 커버리지 히트맵

## 생성 파일

| 파일 | 설명 |
|------|------|
| `output_unified/unified_manifest.json` | 프레임별 통합 신호 + 알람 후보 |
| `output_unified/manifest_coverage.json` | 로드된/누락 manifest 목록 |
| `figures/01_coverage_heatmap.png` | 프레임 x 신호 히트맵 |
| `figures/02_alert_candidates.png` | 알람 후보 요약표 |

## FAQ

**Q. manifest가 대부분 MISSING이에요.**  
A. 해당 번호 노트북(03~07)을 먼저 실행하세요. IPAD만 있어도 keypoint manifest는 생성 가능합니다.

**Q. OCR이 AI Hub만 있고 IPAD OCR이 없어요.**  
A. `ocr_hub`만 병합됩니다. 06 Part 4에서 IPAD 프레임 OCR을 실행하면 `ocr_ipad`가 추가됩니다.

**Q. unified_score는 어떻게 쓰나요?**  
A. 알람/리포트 우선순위용 **데모 가중치**입니다. 현장에서는 04 anomaly score·07 motion_z 비중을 높이세요.
